# Base LLaVA vs HSA-DPO Inference Comparison

This notebook shows the same image/prompt comparison before and after HSA-DPO training. Use the cached section for a fast presentation example from the completed evaluation, and use the live section on a GPU machine when you want to run both models directly.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image as DisplayImage, Markdown, display

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "inference" / "inference_example.py").exists():
    REPO_ROOT = REPO_ROOT.parent

from inference.inference_example import (
    DEFAULT_ADAPTER,
    DEFAULT_BASE_MODEL,
    DEFAULT_EVAL_DIR,
    DEFAULT_IMAGE,
    DEFAULT_PROMPT,
    format_comparison_markdown,
    load_cached_comparison,
    load_metric_summary,
    run_live_comparison,
)

BASE_MODEL = DEFAULT_BASE_MODEL
HSA_DPO_ADAPTER = DEFAULT_ADAPTER
EVAL_DIR = DEFAULT_EVAL_DIR
EXAMPLE_IMAGE = DEFAULT_IMAGE
PROMPT = DEFAULT_PROMPT

## Fast Comparison From Saved Evaluation

This section does not load the models. It reads the outputs generated during the completed evaluation and displays a matched base-vs-HSA-DPO example.

In [ ]:
cached_result = load_cached_comparison(
    EVAL_DIR,
    benchmark="object_halbench",
    example_id="1903",  # set None to automatically choose the first changed example
)

image_path = Path(cached_result.image)
if image_path.exists():
    display(DisplayImage(filename=str(image_path), width=520))
else:
    print(f"Image file is not available on this machine: {cached_result.image}")

display(Markdown(format_comparison_markdown(cached_result)))

## Evaluation Metric Snapshot

These metrics summarize the same before/after model comparison from the full evaluation run.

In [ ]:
rows = load_metric_summary(f"{EVAL_DIR}/comparison/summary.csv")
metric_rows = [
    {
        "benchmark": row["benchmark"],
        "metric": row["metric"],
        "base": row["baseline_value"] or "NA",
        "HSA-DPO": row["our_value"] or "NA",
        "delta": row["delta_vs_baseline"] or "NA",
        "direction": row["direction"],
    }
    for row in rows
    if row["benchmark"] in {"object_halbench", "amber", "pope_adv"}
]
metric_rows

## Live Inference Comparison

Run this section on the GPU machine where `models/llava-v1.5-7b` and the trained adapter are available. It loads the base model first, then loads the HSA-DPO adapter and runs both on the same image/prompt.

In [ ]:
live_image = REPO_ROOT / EXAMPLE_IMAGE
display(DisplayImage(filename=str(live_image), width=520))
print(PROMPT)

In [ ]:
RUN_LIVE = False  # change to True on VastAI/GPU

if RUN_LIVE:
    live_result = run_live_comparison(
        image=EXAMPLE_IMAGE,
        prompt=PROMPT,
        base_model=BASE_MODEL,
        adapter_path=HSA_DPO_ADAPTER,
        temperature=0.0,
        num_beams=1,
        max_new_tokens=256,
    )
    display(Markdown(format_comparison_markdown(live_result)))
else:
    print("Set RUN_LIVE = True on the GPU machine to run live model inference.")